In [1]:
import sqlite3
import pandas as pd
import os
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor

db_path = r"C:\Users\ADMIN\smart agricultural analytics\agriculture.db"

os.makedirs("stage26_models", exist_ok=True)

conn = sqlite3.connect(db_path)

crop = pd.read_sql("SELECT * FROM crop_recommendation_selected", conn)
yield_df = pd.read_sql("SELECT * FROM agri_yield_train_selected", conn)

conn.close()

print("Crop data:", crop.shape)
print("Yield data:", yield_df.shape)

Crop data: (2200, 13)
Yield data: (10000, 11)


In [2]:
crop_features = [
    "humidity", "K", "temperature", "N", "P_K_Ratio",
    "NPK_Total", "Temperature_Rainfall", "rainfall",
    "N_K_Ratio", "Humidity_Rainfall",
    "Temperature_Humidity", "P"
]

X = crop[crop_features]
y = crop["label"]

crop_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100, random_state=42
    ))
])

crop_model.fit(X, y)

joblib.dump(crop_model, "stage26_models/crop_classifier.joblib")

print("Crop classifier saved.")

Crop classifier saved.


In [3]:
yield_features = [
    "NDVI_Temperature", "OrganicMatter_Sand",
    "Temperature_Rainfall", "sand_pct", "NDVI_Rainfall",
    "soil_ph", "organic_matter", "Humidity_Rainfall",
    "Temperature_Humidity", "rainfall"
]

X = yield_df[yield_features]
y = yield_df["yield"]

yield_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        random_state=42
    ))
])

yield_model.fit(X, y)

joblib.dump(yield_model, "stage26_models/yield_predictor.joblib")

print("Yield model saved as an experimental candidate.")

Yield model saved as an experimental candidate.


In [4]:
crop_loaded = joblib.load("stage26_models/crop_classifier.joblib")
yield_loaded = joblib.load("stage26_models/yield_predictor.joblib")

print("Crop model loaded:", type(crop_loaded).__name__)
print("Yield model loaded:", type(yield_loaded).__name__)
print("Files:", os.listdir("stage26_models"))

Crop model loaded: Pipeline
Yield model loaded: Pipeline
Files: ['crop_classifier.joblib', 'yield_predictor.joblib']
